#**1. EXTRAÇÃO E INTEGRAÇÃO DOS DADOS (E)**

O objetivo inicial deste pipeline ETL consiste na extração e unificação dos dados brutos. Como os dados históricos estão fragmentados em 36 planilhas mensais distribuídas ao longo de três anos (2020 a 2022) dentro de diretórios do Google Drive, faremos o mapeamento automatizado desses arquivos. Como validamos previamente que o esquema estrutural de colunas manteve-se idêntico em toda a série temporal, realizamos uma concatenação para consolidar os mais de 5,8 milhões de registros em um único DataFrame unificado, otimizando o início do tratamento.

In [ ]:
import pandas as pd
import glob
import os
from google.colab import drive

# conectando ao Google Drive
drive.mount('/content/drive')

# mapeamento e extração dos arquivos
caminho_base = '/content/drive/MyDrive/trab_rt/Dados_Brutos'
padrao_busca = f'{caminho_base}/*_rt/*_Transferencias.csv'
lista_arquivos = glob.glob(padrao_busca)

lista_dfs = []

for arquivo in lista_arquivos:
    nome_arquivo = os.path.basename(arquivo)
    print(f"Lendo o arquivo: {nome_arquivo}...")
    # base federal: sep=';' e encoding='latin-1'
    df_mes = pd.read_csv(arquivo, sep=';', encoding='latin-1', low_memory=False)
    lista_dfs.append(df_mes)

# concatenação vertical da série histórica completa
print("\nIniciando a concatenação de todos os arquivos...")
df_final = pd.concat(lista_dfs, ignore_index=True)

print("Unificação concluída com sucesso!")
print(f"Total de linhas e colunas na base unificada: {df_final.shape}")

Mounted at /content/drive
Lendo o arquivo: 202001_Transferencias.csv...
Lendo o arquivo: 202010_Transferencias.csv...
Lendo o arquivo: 202002_Transferencias.csv...
Lendo o arquivo: 202003_Transferencias.csv...
Lendo o arquivo: 202004_Transferencias.csv...
Lendo o arquivo: 202005_Transferencias.csv...
Lendo o arquivo: 202006_Transferencias.csv...
Lendo o arquivo: 202007_Transferencias.csv...
Lendo o arquivo: 202008_Transferencias.csv...
Lendo o arquivo: 202009_Transferencias.csv...
Lendo o arquivo: 202011_Transferencias.csv...
Lendo o arquivo: 202012_Transferencias.csv...
Lendo o arquivo: 202101_Transferencias.csv...
Lendo o arquivo: 202102_Transferencias.csv...
Lendo o arquivo: 202103_Transferencias.csv...
Lendo o arquivo: 202104_Transferencias.csv...
Lendo o arquivo: 202105_Transferencias.csv...
Lendo o arquivo: 202106_Transferencias.csv...
Lendo o arquivo: 202107_Transferencias.csv...
Lendo o arquivo: 202108_Transferencias.csv...
Lendo o arquivo: 202109_Transferencias.csv...
Lendo o 

#**2. TRANSFORMAÇÃO DOS DADOS (T)**

##**2.1. Diagnóstico de Qualidade e Perfilamento (Data Profiling)**

Com a base de dados consolidada, iniciamos a etapa de perfilamento para diagnosticar a qualidade e a completude das 36 colunas originais. Uma abordagem puramente baseada em detecção de valores nulos nativos do Pandas (NaN) seria ineficiente neste cenário, uma vez que identificamos a presença de "nulos mascarados" (anomalias de preenchimento inseridas como strings ou códigos estáticos, tais como -1, 'sem informação' ou espaços vazios). Desenvolvemos uma função customizada que itera sobre cada atributo, contabilizando nulos reais e mascarados para calcular com precisão a taxa de utilidade analítica e a cardinalidade de cada variável, gerando uma matriz de qualidade que norteará as decisões de descarte e limpeza.

In [ ]:
import numpy as np

def analisar_colunas(df):
    relatorio = []
    total_linhas = len(df)

    print("Gerando Data Profiling (Perfilamento) das colunas...\n")

    for coluna in df.columns:
        tipo = df[coluna].dtype
        nulos_padrao = df[coluna].isnull().sum()
        # capturando os nulos escondidos
        nulos_disfarcados = df[coluna].isin([-1, '-1', 'sem informação', 'SEM INFORMAÇÃO', 'Sem informação', '']).sum()

        total_ausentes = nulos_padrao + nulos_disfarcados
        pct_ausentes = (total_ausentes / total_linhas) * 100
        pct_preenchidos = 100 - pct_ausentes
        unicos = df[coluna].nunique()

        relatorio.append({
            'Coluna': coluna,
            'Tipo de Dado': tipo,
            'Qtd Nulos/Inválidos': total_ausentes,
            '% Não Nulo (Útil)': round(pct_preenchidos, 2),
            '% Nulo': round(pct_ausentes, 2),
            'Qtd Valores Únicos': unicos
        })

    df_relatorio = pd.DataFrame(relatorio)
    return df_relatorio.sort_values(by='% Nulo', ascending=False).reset_index(drop=True)

tabela_qualidade = analisar_colunas(df_final)
display(tabela_qualidade)

Gerando Data Profiling (Perfilamento) das colunas...



,Coluna,Tipo de Dado,Qtd Nulos/Inválidos,% Não Nulo (Útil),% Nulo,Qtd Valores Únicos
0,DESCRIÇÃO COMPLEMENTAR LOCALIZADOR,object,5838024,0.00,100.00,14
1,SIGLA LOCALIZADOR,object,5837734,0.01,99.99,31
2,CÓDIGO ÓRGÃO SIAFI,float64,5836264,0.03,99.97,12
3,NOME ÓRGÃO,object,5836264,0.03,99.97,12
4,LINGUAGEM CIDADÃ,object,1851047,68.29,31.71,88
5,CÓDIGO PLANO ORÇAMENTÁRIO,object,1282267,78.04,21.96,214
6,NOME ELEMENTO DESPESA,object,1105236,81.07,18.93,22
7,CÓDIGO ELEMENTO DESPESA,object,1105236,81.07,18.93,22
8,CÓDIGO LOCALIZADOR,int64,1105236,81.07,18.93,2227
9,NOME MODALIDADE APLICAÇÃO DESPESA,object,1105236,81.07,18.93,15


## **2.2. Otimização Estrutural e Correção de Tipagem (Data Cleansing)**

A partir do diagnóstico gerado pelo perfilamento, aplicamos regras severas de descarte de dimensionalidade (Drop). Identificamos um bloco de 17 variáveis ligadas exclusivamente à contabilidade interna e auditoria governamental do sistema SIAFI (ex: 'Plano Orçamentário', 'Elemento Despesa'). Visto que esses atributos apresentavam esparsidade crítica próxima a 80% e não possuem relevância para o escopo analítico do nosso Modelo Estrela, sua remoção em lote reduz drasticamente o peso da base na memória RAM. Adicionalmente, realizamos a conversão de tipagem da nossa métrica principal: a coluna VALOR TRANSFERIDO foi interpretada originalmente como texto devido ao padrão brasileiro de pontuação (uso de vírgulas para decimais). O script substitui esses caracteres e força o casting para float64, viabilizando futuras agregações matemáticas corporativas.

In [ ]:
colunas_inuteis = [
    'SIGLA LOCALIZADOR', 'CÓDIGO ÓRGÃO SIAFI', 'NOME ÓRGÃO',
    'DESCRIÇÃO COMPLEMENTAR LOCALIZADOR', 'CÓDIGO PLANO ORÇAMENTÁRIO',
    'CÓDIGO ELEMENTO DESPESA', 'CÓDIGO SUBTÍTULO', 'CÓDIGO GRUPO DESPESA',
    'CÓDIGO LOCALIZADOR', 'CÓDIGO MODALIDADE APLICAÇÃO DESPESA',
    'CÓDIGO UNIDADE GESTORA', 'NOME UNIDADE GESTORA', 'NOME PLANO ORÇAMENTÁRIO',
    'NOME ELEMENTO DESPESA', 'NOME GRUPO DESPESA', 'NOME SUBTÍTULO',
    'NOME LOCALIZADOR'
]

df_final.drop(columns=colunas_inuteis, inplace=True, errors='ignore')

# transformando o valor transferido de texto para float
df_final['VALOR TRANSFERIDO'] = df_final['VALOR TRANSFERIDO'].astype(str).str.replace(',', '.')
df_final['VALOR TRANSFERIDO'] = pd.to_numeric(df_final['VALOR TRANSFERIDO'], errors='coerce')

tabela_qualidade = analisar_colunas(df_final)


display(tabela_qualidade)
print(f"Limpeza profunda concluída! Colunas restantes: {len(df_final.columns)}")


Gerando Data Profiling (Perfilamento) das colunas...



,Coluna,Tipo de Dado,Qtd Nulos/Inválidos,% Não Nulo (Útil),% Nulo,Qtd Valores Únicos
0,LINGUAGEM CIDADÃ,object,1851047,68.29,31.71,88
1,NOME MODALIDADE APLICAÇÃO DESPESA,object,1105236,81.07,18.93,15
2,CÓDIGO MUNICÍPIO SIAFI,float64,313038,94.64,5.36,5571
3,NOME MUNICÍPIO,object,313038,94.64,5.36,5292
4,NOME PROGRAMA,object,26097,99.55,0.45,167
5,CÓDIGO PROGRAMA,object,26044,99.55,0.45,178
6,AÇÃO,object,26044,99.55,0.45,657
7,NOME AÇÃO,object,26059,99.55,0.45,654
8,UF,object,1136,99.98,0.02,28
9,ANO / MÊS,int64,0,100.00,0.00,36


Limpeza profunda concluída! Colunas restantes: 19


##**2.3. Sanitização e Padronização Categórica**
Para mitigar erros decorrentes de falhas de digitação humana e garantir a consistência das dimensões categóricas, implementamos rotinas vetorizadas de higienização de strings. Atributos textuais cruciais como LINGUAGEM CIDADÃ e NOME AÇÃO sofrem a remoção de espaços em branco sobressalentes nas extremidades e são convertidos integralmente para caixa alta, neutralizando duplicidades analíticas artificiais (como diferenciar 'Sem informação' de 'SEM INFORMAÇÃO'). Por fim, todas as variações residuais de lixo textual identificadas no perfilamento são mapeadas uniformemente para termos padronizados de mercado (como 'NÃO INFORMADO' ou 'NI'), garantindo que o Data Warehouse possua strings limpas e previsíveis.  

In [7]:
print("Iniciando a sanitização textual dos dados...")

lixo_textual = ['SEM INFORMAÇÃO', 'NAN', 'NULL', 'NONE', '']

# limpeza e padronização do texto
df_final['LINGUAGEM CIDADÃ'] = df_final['LINGUAGEM CIDADÃ'].astype(str).str.strip().str.upper()
df_final.loc[df_final['LINGUAGEM CIDADÃ'].isin(lixo_textual), 'LINGUAGEM CIDADÃ'] = 'NÃO INFORMADO'

df_final['NOME AÇÃO'] = df_final['NOME AÇÃO'].astype(str).str.strip().str.upper()
df_final.loc[df_final['NOME AÇÃO'].isin(lixo_textual), 'NOME AÇÃO'] = 'NÃO INFORMADO'

df_final['NOME MUNICÍPIO'] = df_final['NOME MUNICÍPIO'].astype(str).str.strip()
df_final.loc[df_final['NOME MUNICÍPIO'].str.upper().isin(['NAN', 'NULL', 'NONE', '']), 'NOME MUNICÍPIO'] = 'NÃO INFORMADO'

df_final['CÓDIGO MUNICÍPIO SIAFI'] = df_final['CÓDIGO MUNICÍPIO SIAFI'].astype(str).str.strip()
df_final.loc[df_final['CÓDIGO MUNICÍPIO SIAFI'].str.upper().isin(['NAN', 'NULL', 'NONE', '']), 'CÓDIGO MUNICÍPIO SIAFI'] = '-1'

df_final['UF'] = df_final['UF'].astype(str).str.strip().str.upper()
df_final.loc[df_final['UF'].isin(lixo_textual), 'UF'] = 'NI'

print("Sanitização textual 100% concluída! Base consistente.")

Iniciando a sanitização textual dos dados...
Sanitização textual 100% concluída! Base consistente.


#**3. CARREGAMENTO E CONSTRUÇÃO DO ESQUEMA ESTRELA (L)**

##**3.1. Isolamento e Geração das Tabelas Dimensão**

Com a base limpa e higienizada, iniciamos a transição para a Modelagem Dimensional em Esquema Estrela (Star Schema). Utilizando operações vetorizadas do Pandas, isolamos os atributos descritivos e removemos suas redundâncias por meio do descarte de duplicatas (drop_duplicates), estruturando as quatro dimensões periféricas mapeadas pelo projeto: Dim_Tempo, Dim_Localidade, Dim_Favorecido e Dim_Programa. Para cada uma dessas tabelas, geramos chaves substitutas numéricas sequenciais. Adicionalmente, injetamos explicitamente o conceito de Membro Desconhecido  em cada dimensão sob a chave estática -1. Com essa prática, garantimos a manutenção da integridade referencial do banco de dados e impedimos a perda de registros financeiros para a Tabela Fato caso ocorra ausência superveniente de metadados.

In [8]:
print("Criando Tabelas Dimensão para o Esquema Estrela...")

# dimensão tempo
df_dim_tempo = df_final[['ANO / MÊS']].drop_duplicates()
df_dim_tempo = df_dim_tempo[df_dim_tempo['ANO / MÊS'] != 'NÃO INFORMADO'].reset_index(drop=True)
df_dim_tempo['ID_TEMPO'] = df_dim_tempo.index + 1
linha_tempo = pd.DataFrame([{'ANO / MÊS': 'NÃO INFORMADO', 'ID_TEMPO': -1}])
df_dim_tempo = pd.concat([df_dim_tempo, linha_tempo], ignore_index=True)

# dimensão localidade
colunas_loc = ['UF', 'CÓDIGO MUNICÍPIO SIAFI', 'NOME MUNICÍPIO']
df_dim_localidade = df_final[colunas_loc].drop_duplicates()
df_dim_localidade = df_dim_localidade[df_dim_localidade['UF'] != 'NI'].reset_index(drop=True)
df_dim_localidade['ID_LOCALIDADE'] = df_dim_localidade.index + 1
linha_loc = pd.DataFrame([{'UF': 'NI', 'CÓDIGO MUNICÍPIO SIAFI': '-1', 'NOME MUNICÍPIO': 'NÃO INFORMADO', 'ID_LOCALIDADE': -1}])
df_dim_localidade = pd.concat([df_dim_localidade, linha_loc], ignore_index=True)

# dimensão favorecido
colunas_fav = ['CÓDIGO FAVORECIDO', 'NOME FAVORECIDO', 'TIPO FAVORECIDO']
df_dim_favorecido = df_final[colunas_fav].drop_duplicates()
df_dim_favorecido = df_dim_favorecido[df_dim_favorecido['CÓDIGO FAVORECIDO'] != '-1'].reset_index(drop=True)
df_dim_favorecido['ID_FAVORECIDO'] = df_dim_favorecido.index + 1
linha_fav = pd.DataFrame([{'CÓDIGO FAVORECIDO': '-1', 'NOME FAVORECIDO': 'NÃO INFORMADO', 'TIPO FAVORECIDO': 'NÃO INFORMADO', 'ID_FAVORECIDO': -1}])
df_dim_favorecido = pd.concat([df_dim_favorecido, linha_fav], ignore_index=True)

# dimensão programa
colunas_prog = ['CÓDIGO PROGRAMA', 'NOME PROGRAMA', 'AÇÃO', 'NOME AÇÃO', 'LINGUAGEM CIDADÃ']
df_dim_programa = df_final[colunas_prog].drop_duplicates()
df_dim_programa = df_dim_programa[df_dim_programa['CÓDIGO PROGRAMA'] != '-1'].reset_index(drop=True)
df_dim_programa['ID_PROGRAMA'] = df_dim_programa.index + 1
linha_prog = pd.DataFrame([{'CÓDIGO PROGRAMA': '-1', 'NOME PROGRAMA': 'NÃO INFORMADO', 'AÇÃO': 'NÃO INFORMADO', 'NOME AÇÃO': 'NÃO INFORMADO', 'LINGUAGEM CIDADÃ': 'NÃO INFORMADO', 'ID_PROGRAMA': -1}])
df_dim_programa = pd.concat([df_dim_programa, linha_prog], ignore_index=True)

print("Dimensões criadas com sucesso!")

Criando Tabelas Dimensão para o Esquema Estrela...
Dimensões criadas com sucesso!


## **3.2. Mapeamento de Chaves e Construção da Tabela Fato**

Para processar o cruzamento estrutural frente ao alto volume de registros sem estourar os limites de memória RAM do Google Colab, aplicamos uma estratégia de junções à esquerda sequenciais (Left Joins). Imediatamente após mapear a Chave Primária de uma dimensão para dentro da Fato, convertendo-a em Chave Estrangeira / FK, realizamos a deleção das colunas de texto redundantes da memória RAM. Eventuais inconsistências ou ausências geradas nas associações são convertidas automaticamente para o ID do Membro Desconhecido (-1). Por fim, eliminamos linhas sem métrica financeira útil e inserimos na primeira posição a chave primária ID_TRANSFERENCIA.

In [9]:
print("Iniciando o mapeamento de chaves na Tabela Fato...")

# tipagem homogênea para os campos de junção geográficos
df_final['CÓDIGO MUNICÍPIO SIAFI'] = df_final['CÓDIGO MUNICÍPIO SIAFI'].astype(str)
df_dim_localidade['CÓDIGO MUNICÍPIO SIAFI'] = df_dim_localidade['CÓDIGO MUNICÍPIO SIAFI'].astype(str)

# junções sequenciais com desalocação de memória imediata para proteção de RAM
df_fato = df_final.merge(df_dim_tempo, on=['ANO / MÊS'], how='left')
df_fato.drop(columns=['ANO / MÊS'], inplace=True)

df_fato = df_fato.merge(df_dim_localidade, on=colunas_loc, how='left')
df_fato.drop(columns=colunas_loc, inplace=True)

df_fato = df_fato.merge(df_dim_favorecido, on=colunas_fav, how='left')
df_fato.drop(columns=colunas_fav, inplace=True)

df_fato = df_fato.merge(df_dim_programa, on=colunas_prog, how='left')
df_fato.drop(columns=colunas_prog, inplace=True)

# isolando apenas Chaves Estrangeiras (FKs) e Métricas Numéricas
colunas_fks = ['ID_TEMPO', 'ID_LOCALIDADE', 'ID_FAVORECIDO', 'ID_PROGRAMA']
colunas_fato = colunas_fks + ['VALOR TRANSFERIDO']
df_fato = df_fato[colunas_fato]

# tratamento e tipagem final das chaves estrangeiras
df_fato[colunas_fks] = df_fato[colunas_fks].fillna(-1).astype(int)

# eliminação de registros corrompidos sem valor financeiro útil
df_fato.dropna(subset=['VALOR TRANSFERIDO'], inplace=True)

# geração da Chave Primária da Tabela Fato (ID_TRANSFERENCIA)
df_fato.insert(0, 'ID_TRANSFERENCIA', range(1, 1 + len(df_fato)))

print("✅ Tabela Fato estruturada com sucesso!")
display(df_fato.head())

Iniciando o mapeamento de chaves na Tabela Fato...
✅ Tabela Fato estruturada com sucesso!


,ID_TRANSFERENCIA,ID_TEMPO,ID_LOCALIDADE,ID_FAVORECIDO,ID_PROGRAMA,VALOR TRANSFERIDO
0,1,1,1,1,1,91799604.24
1,2,1,1,1,2,25996.59
2,3,1,1,1,3,1254445.63
3,4,1,1,1,4,1350804.41
4,5,1,1,1,5,2308.57
